# 01 Bronze ingest

Loads **new** files from `Files/landing` into bronze Delta tables, exactly as they arrived
(all columns as text), plus three columns that say where each row came from:
`_source_file`, `_ingested_at` and `_batch_id`.

A file counts as new when its path and modification time are not yet in `meta.ingestion_log`,
so running the notebook twice loads nothing the second time. Master data files are full
snapshots, so a new snapshot replaces the old one in bronze; the files themselves stay in
landing as history.

In [ ]:
landing = "Files/landing"
batch_id = ""                # the pipeline passes its run id; generated when empty

In [ ]:
import datetime as dt
import uuid

from pyspark.sql import functions as F
from pyspark.sql import types as T

In [ ]:
%run nb_lib_retail

In [ ]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
batch_id = batch_id or dt.datetime.utcnow().strftime("%Y%m%d%H%M%S") + "-" + uuid.uuid4().hex[:6]
ingested_at = dt.datetime.utcnow().replace(microsecond=0)

for schema in ("bronze", "silver", "gold", "meta"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

spark.sql("""
CREATE TABLE IF NOT EXISTS meta.ingestion_log (
    file_path STRING, file_size BIGINT, modified_at TIMESTAMP, source STRING,
    batch_id STRING, row_count BIGINT, ingested_at TIMESTAMP
) USING DELTA
""")

## Find new files

In [ ]:
def list_files(path):
    """Recursive file listing through the Hadoop file system, works on OneLake and locally."""
    jvm = spark._jvm
    p = jvm.org.apache.hadoop.fs.Path(path)
    fs = p.getFileSystem(spark._jsc.hadoopConfiguration())
    if not fs.exists(p):
        return []
    out, it = [], fs.listFiles(p, True)
    while it.hasNext():
        s = it.next()
        out.append((s.getPath().toString(), int(s.getLen()), dt.datetime.utcfromtimestamp(s.getModificationTime() / 1000)))
    return out


def landing_path(path):
    """'abfss://...Files/landing/pos/x.csv' and 'file:/tmp/.../landing/pos/x.csv' both become 'landing/pos/x.csv'."""
    i = path.find("/landing/")
    return path[i + 1:] if i >= 0 else path


def source_of(path):
    for folder, name in [("/pos/", "pos"), ("/webshop/", "webshop"), ("/master/products/", "products"),
                         ("/master/stores/", "stores"), ("/master/fx_rates/", "fx_rates"),
                         ("/master/sales_targets/", "sales_targets")]:
        if folder in path:
            return name
    return None


files = [(p, size, mod, source_of(p)) for p, size, mod in list_files(landing) if source_of(p)]
files = [(landing_path(p), p, size, mod, src) for p, size, mod, src in files]
seen = {(r.file_path, r.modified_at) for r in spark.table("meta.ingestion_log").select("file_path", "modified_at").collect()}
new_files = [f for f in files if (f[0], f[3].replace(microsecond=0)) not in seen]
new_by_source = {}
for f in new_files:
    new_by_source.setdefault(f[4], []).append(f)
print(f"{len(files)} files in landing, {len(new_files)} new: " + ", ".join(f"{k}={len(v)}" for k, v in sorted(new_by_source.items())))

## Load into bronze

In [ ]:
def with_lineage(df):
    return (df.withColumn("_source_file", F.col("_metadata.file_path"))
              .withColumn("_ingested_at", F.lit(ingested_at).cast("timestamp"))
              .withColumn("_batch_id", F.lit(batch_id)))


def read_csv(paths, columns, sep=","):
    schema = T.StructType([T.StructField(c, T.StringType()) for c in columns])
    return with_lineage(spark.read.option("header", True).option("sep", sep).schema(schema).csv(paths))


MASTER_COLUMNS = {
    "products": ["sku", "product_name", "category", "subcategory", "list_price_dkk", "unit_cost_dkk", "launch_date", "discontinued_date"],
    "stores": ["store_code", "store_name", "city", "country", "region", "channel", "currency", "opened_on", "closed_on"],
    "fx_rates": ["rate_date", "currency", "dkk_per_unit"],
    "sales_targets": ["month", "city", "target_dkk"],
}

loaded = []
if "pos" in new_by_source:
    df = read_csv([f[1] for f in new_by_source["pos"]], POS_COLUMNS, sep=";")
    df.write.format("delta").mode("append").saveAsTable("bronze.pos_sales_raw")
    loaded.append(("pos", df))

if "webshop" in new_by_source:
    df = with_lineage(spark.read.schema(WEB_SCHEMA).json([f[1] for f in new_by_source["webshop"]]))
    df.write.format("delta").mode("append").saveAsTable("bronze.web_orders_raw")
    loaded.append(("webshop", df))

for name, cols in MASTER_COLUMNS.items():
    if name in new_by_source:
        newest = max(new_by_source[name], key=lambda f: f[3])
        df = read_csv([newest[1]], cols)
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"bronze.{name}_raw")
        loaded.append((name, df))

## Write the ingestion log

In [ ]:
counts = {}
for name, df in loaded:
    for r in df.groupBy("_source_file").count().collect():
        counts[landing_path(r["_source_file"])] = r["count"]

log_rows = [(rel, size, mod.replace(microsecond=0), src, batch_id, int(counts.get(rel, 0)), ingested_at)
            for rel, _, size, mod, src in new_files]
if log_rows:
    schema = "file_path string, file_size bigint, modified_at timestamp, source string, batch_id string, row_count bigint, ingested_at timestamp"
    spark.createDataFrame(log_rows, schema).write.format("delta").mode("append").saveAsTable("meta.ingestion_log")

print(f"batch {batch_id}: {sum(counts.values())} rows from {len(log_rows)} files")
display(spark.table("meta.ingestion_log").where(F.col("batch_id") == batch_id).orderBy("file_path"))